# Custom REE Elements: Adding New Elements to the Database

The built-in `difflow_ree` database covers 10 commercial REEs (La, Ce, Pr, Nd, Sm, Eu, Gd, Tb, Dy, Y) and 4 extractant systems. However, the lanthanide series contains 15 elements, and real separation processes often involve elements not in the default database (e.g., Ho, Er, Tm, Yb, Lu) or non-REE impurities that co-extract.

This notebook demonstrates how to add custom elements with your own literature data, so you can simulate separations involving any element.

## What you will learn

1. Register a new element with `create_custom_element` and `add_element`
2. Add extraction coefficients for the new element to specific extractants with `add_element_to_extractant`
3. Add separation factor data with `add_pair` and `add_separation_factors`
4. Use the new element in distribution and extraction calculations
5. Clean up custom data when done

In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from difflow_ree import (
    # Custom element creation
    create_custom_element,
    # Database singletons
    get_ree_database,
    get_extractant_database,
    get_sf_database,
    # Convenience functions
    get_element,
    list_ree_elements,
    # Equilibrium
    REEDistribution,
    # Units
    REEExtractor,
    REEExtractorParams,
)
from difflow.streams import make_stream, get_flows

## 1. The built-in database

Before adding anything, let's see what is already available.

In [ ]:
ree_db = get_ree_database()
ext_db = get_extractant_database()
sf_db = get_sf_database()

print("Built-in elements:", ree_db.list_elements())
print("Built-in extractants:", ext_db.list_extractants())
print()

# Show element groups
for group in ("light", "middle", "heavy"):
    print(f"  {group}: {ree_db.list_by_group(group)}")

## 2. Adding Holmium to the element database

Holmium (Ho, Z=67) sits between Dysprosium and Erbium in the lanthanide series. Its physical properties are well-established constants from standard references.

**Important:** The physical properties below (atomic weight, ionic radius, density, melting point) are standard reference values. The extraction coefficients in later cells are *placeholders* to demonstrate the API. In a real application, you should replace them with values from your own experimental data or published correlations for your specific system.

In [ ]:
# Create Holmium from standard reference data
# Sources: CRC Handbook (physical properties), Shannon 1976 (ionic radius)
ho = create_custom_element(
    symbol="Ho",
    name="Holmium",
    atomic_number=67,
    atomic_weight=164.930,   # g/mol
    ionic_radius_pm=90.1,    # pm, CN=6, 3+ oxidation state (Shannon 1976)
    density=8.795,           # g/cm3
    melting_point=1734,      # K
    group="heavy",           # Ho behaves as a heavy REE
    oxide_formula="Ho2O3",
    oxide_mw=377.86,         # g/mol
    price_usd_kg=60.0,      # approximate market price, varies
)

# Register it
ree_db.add_element("Ho", ho)

print(f"Added: {ho.name} ({ho.symbol}), Z={ho.atomic_number}")
print(f"  Atomic weight: {ho.atomic_weight} g/mol")
print(f"  Ionic radius:  {ho.ionic_radius_pm} pm")
print(f"  Group:         {ho.group}")
print()
print("Heavy REEs now:", ree_db.list_by_group("heavy"))

## 3. Adding extraction coefficients for PC88A

Now we need pH and temperature coefficients so the distribution model can compute D values for Ho. These are extractant-specific empirical correlations, so we add them only to the extractants we have data for.

The model is: `log10(D) = a + b*pH + c*pH^2`, with a temperature correction `d*(1/T - 1/T_ref)`.

**The values below are illustrative placeholders.** In practice, you would obtain these by fitting published D vs. pH data for Ho with your extractant system (see notebook `21_custom_extractants.ipynb` for a calibration example).

In [ ]:
# Add Ho coefficients to PC88A only
# PLACEHOLDER VALUES - replace with your literature data!
#
# These follow the lanthanide contraction trend in the PC88A data:
#   Tb: a=-6.52, b=2.82   Dy: a=-6.32, b=2.90
# Ho (Z=67) should sit just above Dy, continuing the ~0.18/0.07 step:
ext_db.add_element_to_extractant(
    "PC88A",
    "Ho",
    ph_coefficients={
        "a": -6.15,    # slightly less negative than Dy (-6.32)
        "b": 2.95,     # slightly steeper than Dy (2.90)
        "c": 0.010,    # same quadratic term as series
    },
    temperature_coefficient=-2350,  # K, continuing trend from Dy (-2300)
)

# Verify: Ho is now in PC88A but NOT in other extractants
pc88a = ext_db.get("PC88A")
print("Ho in PC88A:", "Ho" in pc88a.ph_coefficients)
print("Ho in D2EHPA:", "Ho" in ext_db.get("D2EHPA").ph_coefficients)
print()
print(f"PC88A now covers: {sorted(pc88a.ph_coefficients.keys())}")

## 4. Adding separation factor data

Separation factors quantify how well two elements can be separated. You can add individual pairs to existing extractants, or create a complete entry for a new extractant.

In [ ]:
# Add Ho separation factor pairs to PC88A
# Convention: "heavier_lighter", SF = D_heavier / D_lighter
# PLACEHOLDER VALUES - replace with your literature data!
sf_db.add_pair("PC88A", "Ho_Dy", 1.4, adjacent=True, stages_99=20)
sf_db.add_pair("PC88A", "Y_Ho", 0.9, adjacent=True)

# Non-adjacent group pair
sf_db.add_pair("PC88A", "Ho_Gd", 2.8, adjacent=False)

# Query the data back
print("SF(Ho/Dy) with PC88A:", sf_db.get_sf("PC88A", "Ho_Dy"))
print("SF(Y/Ho)  with PC88A:", sf_db.get_sf("PC88A", "Y_Ho"))
print("Stages for 99% Ho/Dy:", sf_db.get_stages_needed("PC88A", "Ho_Dy"))

## 5. Using Ho in distribution calculations

With the element registered and extraction coefficients added, Ho works just like any built-in element in the distribution model.

In [ ]:
# Create distribution model including Ho
dist = REEDistribution(
    extractant="PC88A",
    elements=("Gd", "Dy", "Ho", "Y"),
    concentration=0.5,
)

# D values at pH 2.0 (low enough to see meaningful differences)
print("Distribution coefficients with PC88A at pH 2.0, 25 C:")
print("-" * 50)
D_all = dist.get_D_all(pH=2.0, T=298.15)
for elem, D in D_all.items():
    print(f"  D({elem:>2}) = {float(D):8.4f}")

print()

# Separation factors from D values
D_Ho = float(D_all["Ho"])
D_Dy = float(D_all["Dy"])
D_Gd = float(D_all["Gd"])
D_Y  = float(D_all["Y"])
print("Separation factors (should show Ho > Dy > Y > Gd):")
print(f"  SF(Ho/Dy) = D(Ho)/D(Dy) = {D_Ho/D_Dy:.2f}")
print(f"  SF(Ho/Gd) = D(Ho)/D(Gd) = {D_Ho/D_Gd:.2f}")
print(f"  SF(Y/Ho)  = D(Y)/D(Ho)  = {D_Y/D_Ho:.2f}")

## 6. Using Ho in a multi-stage extraction unit

Custom elements also work with the unit operation models. Here we run a 5-stage extraction of a Gd/Ho/Y mixture with PC88A.

In [ ]:
# Create extractor at pH 2.0 where there is meaningful selectivity
params = REEExtractorParams(
    n_stages=5,
    extractant="PC88A",
    elements=("Gd", "Ho", "Y"),
    pH=2.0,
    include_loading=False,  # no loading isotherm data for Ho
)
extractor = REEExtractor(params)

# Feed stream: mixed Gd/Ho/Y solution
feed = make_stream(
    flows={"H2O": 10.0, "Gd": 0.02, "Ho": 0.01, "Y": 0.015},
    T=298.15, P=101325.0,
)

# Organic solvent (S/F ~ 1.0 by volume)
solvent = make_stream(
    flows={"Organic": 10.0, "Gd": 0.0, "Ho": 0.0, "Y": 0.0},
    T=298.15, P=101325.0,
)

# Run extraction
raffinate, extract, info = extractor(feed, solvent)

# Results
feed_flows = get_flows(feed)
ext_flows = get_flows(extract)

print("Extraction results (5 stages, PC88A, pH 2.0, S/F=1.0):")
print("-" * 55)
print(f"{'Element':<10} {'Feed (mol/s)':<15} {'Extract':<15} {'Recovery %'}")
print("-" * 55)
for elem in ["Gd", "Ho", "Y"]:
    f_val = float(feed_flows[elem])
    e_val = float(ext_flows[elem])
    rec = (e_val / f_val) * 100 if f_val > 0 else 0
    print(f"{elem:<10} {f_val:<15.4f} {e_val:<15.4f} {rec:.1f}")

## 7. Gradients work through custom elements

Because the custom element uses the same differentiable model, JAX automatic differentiation works as expected. Here we compute the gradient of Ho recovery with respect to pH.

In [ ]:
from jax import grad

def ho_recovery(pH):
    """Compute Ho recovery as a function of extraction pH."""
    params = REEExtractorParams(
        n_stages=5,
        extractant="PC88A",
        elements=("Gd", "Ho", "Y"),
        pH=pH,
        include_loading=False,
    )
    extractor = REEExtractor(params)

    feed = make_stream(
        flows={"H2O": 10.0, "Gd": 0.02, "Ho": 0.01, "Y": 0.015},
        T=298.15, P=101325.0,
    )
    solvent = make_stream(
        flows={"Organic": 10.0, "Gd": 0.0, "Ho": 0.0, "Y": 0.0},
        T=298.15, P=101325.0,
    )

    _, extract, _ = extractor(feed, solvent)
    ext_flows = get_flows(extract)
    return ext_flows["Ho"] / 0.01  # recovery fraction

# Gradient of recovery w.r.t. pH
d_recovery_d_pH = grad(ho_recovery)

pH_test = 2.0
rec = float(ho_recovery(pH_test))
drec = float(d_recovery_d_pH(pH_test))

print(f"At pH {pH_test}:")
print(f"  Ho recovery:     {rec:.4f} ({rec*100:.1f}%)")
print(f"  d(recovery)/dpH: {drec:.4f}")
print(f"  -> Increasing pH by 0.1 would change recovery by ~{drec*0.1*100:.1f}%")

## 8. Correcting or removing custom data

If you need to update coefficients (e.g., after getting better literature data), remove and re-add. All custom data modifications are runtime-only and do not affect the YAML files on disk.

In [ ]:
# Update extraction coefficients: remove and re-add
ext_db.remove_element_from_extractant("PC88A", "Ho")
ext_db.add_element_to_extractant(
    "PC88A", "Ho",
    ph_coefficients={"a": -6.18, "b": 2.96, "c": 0.011},  # revised values
    temperature_coefficient=-2380,
)
print("Updated Ho coefficients for PC88A")

# Update element properties (e.g., corrected price)
updated_ho = create_custom_element(
    symbol="Ho", name="Holmium", atomic_number=67,
    atomic_weight=164.930, ionic_radius_pm=90.1, density=8.795,
    melting_point=1734, group="heavy", oxide_formula="Ho2O3",
    oxide_mw=377.86, price_usd_kg=75.0,  # updated price
)
ree_db.update_element("Ho", updated_ho)
print(f"Updated Ho price: ${ree_db.get('Ho').price_usd_kg}/kg")

# Remove separation factor pair and re-add
sf_db.remove_pair("PC88A", "Ho_Dy")
sf_db.add_pair("PC88A", "Ho_Dy", 1.5, stages_99=18)  # revised value
print(f"Updated SF(Ho/Dy): {sf_db.get_sf('PC88A', 'Ho_Dy')}")

## 9. Cleanup

Remove Ho from all databases to restore the original state. Since the databases are singletons, this matters if other code in your session relies on the default element list.

In [ ]:
# Remove Ho from extractant coefficients
ext_db.remove_element_from_extractant("PC88A", "Ho")

# Remove Ho separation factor pairs
for pair in ["Ho_Dy", "Y_Ho", "Ho_Gd"]:
    try:
        sf_db.remove_pair("PC88A", pair)
    except KeyError:
        pass

# Remove Ho from element database
ree_db.remove_element("Ho")

print("Restored databases:")
print("  Elements:", ree_db.list_elements())
print("  Heavy REEs:", ree_db.list_by_group("heavy"))
print("  PC88A elements:", sorted(ext_db.get("PC88A").ph_coefficients.keys()))

## Summary

This notebook demonstrated the full workflow for extending the REE database with custom elements:

| Step | Function | Purpose |
|------|----------|---------|
| Create element | `create_custom_element(...)` | Build an `REEElement` from physical properties |
| Register element | `ree_db.add_element(symbol, elem)` | Make it available to all database queries |
| Add extraction data | `ext_db.add_element_to_extractant(...)` | Provide pH/temperature coefficients for specific extractants |
| Add SF data | `sf_db.add_pair(...)` | Provide separation factors for specific element pairs |
| Update data | `remove_*` then re-add | Correct values when better literature data becomes available |

All modifications are runtime-only: they do not alter the YAML files on disk, so restarting Python restores the defaults.

For creating entirely new extractants (rather than adding elements to existing ones), see notebook `21_custom_extractants.ipynb`.